# Real Option Chains — data pull, quality gate, and the kill criterion

Run this in **Colab**. It cannot run in the Claude Code sandbox: that environment's
egress policy blocks every options vendor (Databento, ThetaData, Polygon, Alpaca,
ORATS, CBOE DataShop, DoltHub all return 403 on CONNECT). Colab is not restricted.

**What this notebook decides.** `STRATEGY.md` pre-registered a kill criterion:

> Abandon the variance-risk-premium thesis if frictionless straddle capture
> measured on real chains is below 3%.

Everything before that is plumbing. Step 5 is the decision, and it is worth
running before any strategy work, because if real capture is below 3% then the
whole VRP direction is dead and no amount of implementation quality saves it.

**Order matters.** The chain is quality-gated *before* any strategy touches it.
That ordering already caught two silent bugs on synthetic data; real chains are
far dirtier.

## 0. Setup

Clone the repo and install. Nothing here needs a paid API key yet — step 2 uses
the free DoltHub dataset.

In [ ]:
!git clone -q https://github.com/parsiqman/flow-signal.git 2>/dev/null || true
%cd flow-signal
!pip install -q pandas numpy matplotlib

import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

from data import chain, loaders, quality, schema
print('ready')

## 1. Where the data comes from

| Source | History | Cost | Notes |
|---|---|---|---|
| **DoltHub `post-no-preference/options`** | ~2019 → now | **free** | EOD chains, bids/asks/greeks, ~2100 symbols. Covers Mar-2020, 2022, Aug-2024 — three vol regimes. **Start here.** |
| ORATS near-EOD | 2007 → now | paid | The only cheap route to 2008. Buy only if the free data survives step 5. |
| CBOE DataShop | deep | paid per dataset | Authoritative, pricier. |
| Databento OPRA | ~2021 → now | pay-per-GB, $125 free credit | Tick-level. Does **not** reach 2008 — check coverage before buying for tail work. |

The staged logic: **free data answers the kill question**. Only spend money if
the premium is actually there.

## 2. Option A — free DoltHub pull

Dolt is a SQL database with git semantics. The clone is a few GB, so this is the
slowest cell in the notebook.

In [ ]:
# Install the dolt CLI and clone the free options database.
!sudo bash -c 'curl -L https://github.com/dolthub/dolt/releases/latest/download/install.sh | bash' 2>/dev/null || \
 !curl -L https://github.com/dolthub/dolt/releases/latest/download/install.sh | sudo bash

!dolt clone post-no-preference/options /content/options_db
%cd /content/options_db
!dolt sql -q "show tables"
%cd /content/flow-signal

In [ ]:
# Pull a manageable slice: liquid names, a few years, EOD chains.
# Widen UNIVERSE and the date range once the pipeline is working end to end.
UNIVERSE = ['SPY','QQQ','IWM','AAPL','MSFT','NVDA','AMZN','META','GOOGL','TSLA',
            'JPM','XOM','WMT','UNH','HD','BAC','PFE','CSCO','INTC','T']
START, END = '2020-01-01', '2024-12-31'

tickers = "','".join(UNIVERSE)
query = f"""
select o.date, o.act_symbol, o.expiration, o.strike, o.call_put,
       o.bid, o.ask, o.vol, o.delta,
       s.close as underlying_price
from option_chain o
join stock_history s
  on s.act_symbol = o.act_symbol and s.date = o.date
where o.act_symbol in ('{tickers}')
  and o.date between '{START}' and '{END}'
"""

%cd /content/options_db
!dolt sql -r csv -q "{query}" > /content/raw_chain.csv
%cd /content/flow-signal

raw = pd.read_csv('/content/raw_chain.csv')
print(f'{len(raw):,} raw rows')
raw.head(3)

### Column mapping

DoltHub's schema is already in `loaders.DOLTHUB_OPTIONS`. If your export's
columns differ, pass your own `column_map` — that's the only thing a new vendor
should ever require.

In [ ]:
cmap = dict(loaders.DOLTHUB_OPTIONS)
chain_df = loaders.from_frame(raw, cmap, strict=False)
print(schema.describe(chain_df))
chain_df.head(3)

## 2b. Option B — a vendor CSV you already have

Skip section 2 entirely and point this at your file. `vendor=` accepts
`dolthub`, `orats` or `polygon`; for ORATS exports that put calls and puts on
one row, use `loaders.load_orats_pair(raw)` instead.

In [ ]:
# chain_df = loaders.load_csv('/content/my_chains.csv', vendor='orats', strict=False)
# print(schema.describe(chain_df))

## 3. The quality gate — run this before anything else

Real chains are filthy and every defect below fails **silently**. In descending
order of damage:

1. **Zero bids treated as sellable** — no buyer at any price, but `(0+ask)/2`
   looks tradeable. Worst on exactly the far-OTM wings a defined-risk strategy uses.
2. **Survivorship** — a universe of "names optionable today" has deleted every
   company that went bankrupt or was acquired. For a *short-vol* strategy that
   bias is favourable and large: the names that blew up are the missing ones.
3. **Adjusted options** — post-split contracts deliver non-standard amounts. No
   pricing model applies.
4. **Stale quotes**, **arbitrage violations**, **timestamp misalignment**.

`assert_usable` raises rather than warns, deliberately: a warning at the top of
a long run is read exactly once.

In [ ]:
report = quality.check_chain(chain_df, min_years=3.0)
print(report.summary())
print()
report.to_frame()

In [ ]:
# Inspect anything that blocked, then decide: fix the mapping, or get better data.
for f in report.findings:
    if f.severity == 'block':
        print(f)

# Once satisfied, this raises if the data is still unusable:
# quality.assert_usable(chain_df, min_years=3.0)

In [ ]:
# Keep only rows a seller could genuinely trade.
clean = quality.clean(chain_df, require_bid=True, max_relative_spread=1.0)
print(f'{len(chain_df):,} -> {len(clean):,} usable rows '
      f'({len(clean)/max(len(chain_df),1):.1%})')
print(schema.describe(clean))

## 4. Build the panel

Collapses contracts into `day, name, spot, atm_iv, days_to_earnings` — byte-identical
in shape to what `options_alpha.synthetic.generate_market` emits, so real and
synthetic data are interchangeable and every existing test applies to both.

Implied vol is interpolated to a **constant 30-day tenor in total variance**.
Listed expiries jump around (28 days, then 21, then 35); feeding that raw into a
richness signal measures the expiry calendar as much as the market. Interpolating
in vol rather than variance is a common shortcut that biases the term structure.

In [ ]:
panel = chain.build_panel(clean, earnings=None, target_days=30)
print(panel.shape)
print(panel.head())

ax = (panel.groupby('date')['atm_iv'].mean()
      .plot(figsize=(13,4), title='Cross-sectional mean 30-day ATM implied vol'))
ax.set_ylabel('implied vol'); ax.grid(alpha=.3)

## 5. THE KILL CRITERION

Sell an ATM straddle on every (date, name) at the listed expiry nearest 30 days,
**at the bid** (what a seller actually receives), hold to expiration, compare
premium against payout.

Frictionless and deliberately naive. This is not a strategy — it is the question
*"is there a premium here at all?"*

`STRATEGY.md` commits to abandoning the thesis below **3%**. Honour that.

In [ ]:
trades = chain.measure_vrp(clean, holding_days=30)
summary = chain.vrp_summary(trades)
for k, v in summary.items():
    print(f'{k:20} {v}')

In [ ]:
if len(trades):
    by_year = (trades.assign(year=pd.to_datetime(trades['date']).dt.year)
               .groupby('year')
               .apply(lambda g: g['pnl'].sum()/g['premium'].sum(),
                      include_groups=False))
    ax = by_year.plot(kind='bar', figsize=(11,4),
                      title='Straddle capture by year (negative = sellers lost)')
    ax.axhline(0.03, color='r', ls='--', label='3% kill threshold')
    ax.axhline(0, color='k', lw=.8); ax.legend(); ax.grid(alpha=.3)

### How to read this

- **Capture below 3%** → the pre-registered kill criterion fires. Record it in the
  hypothesis registry and stop. That is a real, useful, money-saving result — the
  most likely honest outcome of any strategy research is "smaller than it looked".
- **A single bad year** is expected. Short vol loses in vol shocks; that is what
  it is being paid for. The question is whether the *full-sample* capture clears
  the bar with the crisis included, not whether every year is positive.
- **Capture far above 15%** → suspect the data before celebrating. Check the
  survivorship finding in step 3 first.

## 6. Run the strategy on real data

Only if step 5 cleared. The panel is schema-compatible, so the existing backtest
runs unchanged — swap `generate_market()` for `panel`.

In [ ]:
from options_alpha.backtest import run_backtest
from options_alpha.strategy import StrategyConfig
from options_alpha.research import split_panel, walk_forward

bt_panel = panel[['day','name','spot','atm_iv','days_to_earnings']].copy()
bt_panel.attrs['shock_days'] = []

wf = walk_forward(bt_panel)
print('frozen:', {k: getattr(wf['frozen_config'], k)
                  for k in ('short_delta','wing_width_frac','profit_target','both_sides')})
print('\nOUT OF SAMPLE (the only number that counts):')
for k, v in wf['oos_stats'].items():
    print(f'  {k:26} {v}')

## 7. Through the validation gauntlet

The strategy result means nothing until it clears the search that produced it.
On synthetic data this blocked our own strategy at deflated Sharpe 0.81 vs a
0.95 bar — expect real data to be *harder*, not easier.

In [ ]:
from lab import validation
from lab.protocol import returns_from_curve

oos_returns = returns_from_curve(wf['oos_curve']['marked_equity'])
n_trials = len(wf['in_sample_table'])

g = validation.run_gauntlet(
    candidate='vrp-001-real',
    oos_returns=oos_returns,
    n_trials=n_trials,
    trial_sharpes=wf['in_sample_table']['sharpe'].to_numpy())
print(g.to_frame().to_string(index=False))
print()
print(g.summary())

## 8. What to do with the answer

**Blocked** → that is the platform working. Do **not** loosen the threshold, and
do **not** search harder: more searching raises the bar the winner must clear.
The two legitimate routes are a genuinely better strategy, or more data.

**Cleared** → next is buying deeper history (ORATS, 2007+) so 2008 is in the
sample, then a full paper-trading quarter. `pipeline.py` will refuse to promote
to live without it.

Either way, record the outcome in the hypothesis registry so the trial count
stays honest for whatever gets tested next.